In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']
# 第一节 交流电的向量计算
# 定负数 用j 
Z1 = 3 + 4j
magnitude = np.abs(Z1) ##用abs算出幅值
phase_angle = np.angle(Z1 , deg = True) ## 用angle算出角度 加上 deg = True 就是角度制
print(magnitude, phase_angle)
#实战
Z2 = 8+6j
magnitude_Z2 = np.abs(Z2)
phase_angle_Z2 = np.angle(Z2 , deg = True)
print(magnitude_Z2, phase_angle_Z2)
## 复杂电路矩阵求解
nodal1 = np.array([[3,-1] , [-1,2]])
I = np.array([5,0]).reshape(-1,1)
print(I)
print(nodal1)
Voltage = np.linalg.solve(nodal1, I)
print(Voltage)
##物理可视化
plt.figure(figsize = (12,10))
f = np.linspace(10,100,500)
Xl = 2 * np.pi * f * 0.1
Xc = 1 / (2 * np.pi * f * 100e-6)
Z = 10 + 1j * (Xl - Xc) #虚数单位相乘 要有个乘号 
plt.subplot(1,3,1)
plt.plot(f , Xl)
plt.xlabel('f的大小')
plt.ylabel('Xl')
plt.subplot(1,3,2)
plt.plot(f , Xc)
plt.xlabel('f的大小')
plt.ylabel('Xc')
plt.subplot(1,3,3)
plt.plot(f , np.abs(Z)) ## plt画图 不可以直接画负数 你要选择画幅值还是角度
plt.xlabel('f的大小')
plt.ylabel('Z')
plt.show()

In [ ]:
import pandas as pd

# 模拟一份凌乱的原始电表数据
data = {
    '时间': ['2026-03-15 01:05:00', '2026-03-15 01:45:00', '2026-03-15 02:10:00'],
    '负荷(kW)': [100, 120, 110]
}
df = pd.DataFrame(data)

# 第一步：把字符串变成真正的时间格式，并设为索引 (这步是重采样的强制前置条件)
df['时间'] = pd.to_datetime(df['时间'])
df.set_index('时间', inplace=True)

# 第二步：按小时('1H')重采样，并计算这一个小时内所有记录的平均值(.mean())
clean_df = df.resample('1h').mean()
print(clean_df)

In [ ]:
#时间序列 (resample , pd.to_datetime)
import pandas as pd

# 1. 模拟凌乱的时间戳（故意跨越两天，且时间点极其不规律）
dirty_times = [
    '2026-03-15 08:12:00', '2026-03-15 14:45:00', '2026-03-15 23:05:00',
    '2026-03-16 02:10:00', '2026-03-16 11:22:00', '2026-03-16 18:50:00'
]

# 2. 模拟变压器温度（单位：℃）
temperatures = [68.5, 82.1, 75.0, 71.2, 84.5, 79.8]

# 3. 直接在创建 DataFrame 时，把时间列转化为 DatetimeIndex 并设为索引
temp_df = pd.DataFrame({'温度(℃)': temperatures}, index=pd.to_datetime(dirty_times))

print("--- 清洗前的凌乱日志 ---")
print(temp_df)
#开始重采样
clean_df1 = temp_df.resample('1D').max()
print('-------清洗后的数据-------')
print(clean_df1)

In [ ]:
#物理除噪
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
plt.rcParams['axes.unicode_minus'] = False
# 1. 生成干净的 50Hz 基波 (时长0.1秒)
t = np.linspace(0, 0.1, 500)
clean_wave = 311 * np.sin(2 * np.pi * 50 * t)

# 2. 制造高频噪声 (比如 1000Hz 的电磁干扰)
noise = 50 * np.sin(2 * np.pi * 1000 * t)

# 3. 叠加得到“脏波形”
dirty_wave = clean_wave + noise

# 绘制脏波形看看长什么样
plt.plot(t, dirty_wave, label='Dirty Waveform', color='red')
plt.legend()
plt.show()
##开始滤波
# 先算采样频率
fs = 5000
# 再算截止频率 就是我们要截止的频率
fc = 150
# Wn = 截止 / 采样的一半
Wn = fc/(fs /2 )
b , a = signal.butter(4 , Wn , btype = 'low')
clean_wave = signal.filtfilt(b , a , dirty_wave)
plt.plot(t, clean_wave, label='Cleaned Waveform', color='red')
plt.legend()
plt.show()

In [ ]:
#FFT频域特征提取
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
plt.rcParams['axes.unicode_minus'] = False

# --- 模拟变电站传回的波形 ---
fs = 2000
N = 1000
t = np.linspace(0, N/fs, N, endpoint=False)
grid_wave = 220 * np.sin(2 * np.pi * 50 * t) + 80 * np.sin(2 * np.pi * 150 * t)

# --- 请在这里写下你的 FFT 破解代码 ---
# 1. 生成频率轴 freq_x:
freq_x = fftfreq(N, 1/fs)

# 2. 生成幅值轴 amp_y:
amp_y = np.abs(fft(grid_wave))

# -----------------------------------

# 画图验证你的破解结果
plt.figure(figsize=(10, 4))
plt.plot(freq_x[:N//2], amp_y[:N//2], color='purple')
plt.title('变电站波形 FFT 频谱分析 (寻找150Hz谐波)')
plt.xlabel('频率 (Hz)')
plt.ylabel('能量幅值')
plt.grid()
plt.show()

In [1]:
import torch

# 1. 检查 PyTorch 版本
print(f"PyTorch 版本: {torch.__version__}")

# 2. 检查 CUDA (GPU加速) 是否可用
cuda_available = torch.cuda.is_available()
print(f"CUDA 是否可用: {cuda_available}")

if cuda_available:
    # 3. 检查当前使用的 GPU 设备名称
    print(f"当前 GPU 设备: {torch.cuda.get_device_name(0)}")
else:
    print("当前正在使用 CPU 运行。")

# 4. 进行一个简单的张量运算测试
x = torch.rand(5, 3)
print("\n测试张量生成:")
print(x)

PyTorch 版本: 2.10.0+cu126
CUDA 是否可用: True
当前 GPU 设备: NVIDIA GeForce RTX 4060 Laptop GPU

测试张量生成:
tensor([[0.7936, 0.5791, 0.1492],
        [0.5327, 0.8889, 0.7675],
        [0.0564, 0.2109, 0.0956],
        [0.0378, 0.0364, 0.6783],
        [0.5757, 0.7083, 0.7975]])
